# How much to trust a detection, per place

A hot pixel is not always a fire. `trust()` (and `facts.place()` for an AI)
says how far a detection at a place can be read as a vegetation fire, from
two sources:

- the **static-source mask**, learnt from the archives: places hot year
  after year (steelworks, gas flares, volcanoes);
- the **land cover** around (ESA WorldCover): water and snow cannot burn,
  built-up land and bare rock are hot for other reasons.

A part without its source does not count, and with neither the answer is
**not measured** (`None`), never 0 and never 1.

This notebook runs offline on the crops of the benchmark (`benchmarks/data`):
no key, no download.

In [1]:
import json
from pathlib import Path

from firms_client import StaticMask, WorldCover, facts, trust
from firms_client.reliability import RELIABLE_FROM, detection_table

DATA = Path.cwd().parent / "benchmarks" / "data"


def mask(case):
    return StaticMask.load(DATA / case / "mask.json")


def land(case):
    return WorldCover(str(DATA / case / "worldcover"))

## Four places

The Taranto steelworks, the wooded slopes above Palermo that burnt in July
2023, a burnt field near Aragona (Sicily), and the open sea, where nothing
is known.

In [2]:
places = [
    ("Taranto, the steelworks", 40.515, 17.195, {"mask": mask("taranto")}),
    ("Palermo, the 2023 fires (EFFIS)", 38.046, 13.3195, {"land": land("palermo-2023")}),
    ("Aragona, a burnt field (EFFIS)", 37.445, 13.63, {"land": land("aragona-2024")}),
    ("Tyrrhenian Sea", 40.0, 11.0, {}),
]
for name, lat, lon, sources in places:
    r = trust(lat, lon, **sources)
    score = "not measured" if r.score is None else f"{r.score:.2f}"
    print(f"{name:32} {score:>13}  reliable: {r.reliable}")
    for why in r.reasons:
        print(f"{'':34}- {why}")
print(f"\nreliable from {RELIABLE_FROM}")

Taranto, the steelworks                   0.00  reliable: False
                                  - static land source: 537 fire-days over 13 years in the archives


Palermo, the 2023 fires (EFFIS)           1.00  reliable: True
Aragona, a burnt field (EFFIS)            1.00  reliable: True
Tyrrhenian Sea                    not measured  reliable: False
                                  - not measured: no static mask and no land cover here

reliable from 0.5


## The same, as facts for an AI

`facts.place()` returns the answer as data: every number with its sources,
licence and notes. A model writes the sentence from it, and `check_text()`
finds any number the model added.

In [3]:
print(json.dumps(facts.place(37.445, 13.63, land=land("aragona-2024")), indent=2))

{
  "answer": "trust_at",
  "latitude": 37.445,
  "longitude": 13.63,
  "measured": true,
  "reliability": {
    "score": 1.0,
    "reliable": true,
    "reliable_from": 0.5,
    "parts": {
      "static": null,
      "land": 1.0
    },
    "reasons": []
  },
  "land_cover": {
    "cropland": 0.876,
    "grassland": 0.123
  },
  "radius_m": 500.0,
  "sources": {
    "static_mask": null,
    "land_cover": "ESA WorldCover 2021 v200"
  },
  "licence": [
    "We acknowledge the use of data and imagery from LANCE FIRMS operated by NASA's Earth Science Data and Information System (ESDIS) with funding provided by NASA Headquarters.",
    "ESA WorldCover 10 m 2021 v200, \u00a9 ESA WorldCover project / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium. CC BY 4.0."
  ],
  "notes": [
    "A detection is a hot pixel seen from space, not an alert: fires are confirmed on the ground, and warnings come from Civil Protection."
  ]
}


## What FIRMS sees, by fire size

The other half of trust: a fire that FIRMS did not see. On the fires EFFIS
mapped in Italy (2022-2025), the share VIIRS saw by size class.

In [4]:
table = detection_table()
print(table["product"], table["country"], "- truth:", table["truth"])
for row in table["classes"]:
    p = row["periods"]["2022-2025"]
    top = "" if row["to_ha"] is None else f"{row['to_ha']:.0f}"
    share = p["seen"] / p["perimeters"]
    print(f"{row['from_ha']:>6.0f}-{top:<6} ha  seen {share:.0%} of {p['perimeters']:,} fires")

viirs-snpp Italy - truth: EFFIS burnt-area database, https://maps.effis.emergency.copernicus.eu/effis ms:modis.ba.poly, © European Union, CC BY 4.0
     0-30     ha  seen 20% of 3,959 fires
    30-100    ha  seen 58% of 1,203 fires
   100-300    ha  seen 76% of 471 fires
   300-1000   ha  seen 87% of 152 fires
  1000-       ha  seen 96% of 27 fires
